# OpenCabinet Diffusion Policy Notebook

End-to-end notebook for the state-based diffusion policy (1D Conv U-Net).

Flow:
1. Environment bootstrap (local or Colab)
2. Download dataset
3. Run 05b augmentation (required by default)
4. Train diffusion policy
5. Evaluate and render rollout video

## 0) Runtime Mode

This notebook works in both:
- Local Jupyter (inside this repo)
- Google Colab (GPU runtime recommended)

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("IN_COLAB:", IN_COLAB)

# Move into cabinet_door_project when running locally from repo root.
if Path.cwd().name != "cabinet_door_project":
    if (Path.cwd() / "cabinet_door_project").exists():
        os.chdir("cabinet_door_project")

print("CWD:", Path.cwd())

## 1) Colab Bootstrap (Run Only on Colab)

If you are in Colab, edit `REPO_URL` and run this cell once.
If you are local, this cell safely does nothing.

In [ ]:
REPO_URL = "https://github.com/<your-user>/<your-repo>.git"
REPO_ROOT = "/content/cs188-default-project"

if IN_COLAB:
    if "<your-user>" in REPO_URL or "<your-repo>" in REPO_URL:
        raise ValueError("Edit REPO_URL to your actual repo URL before running this cell.")

    if not os.path.exists(REPO_ROOT):
        !git clone $REPO_URL $REPO_ROOT

    %cd $REPO_ROOT
    !python cabinet_door_project/99_colab_setup.py --verify
    %cd cabinet_door_project
else:
    print("Skipping Colab bootstrap (not in Colab).")

## 2) Hardware Check

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
print("mps available:", torch.backends.mps.is_available())

## 3) Download OpenCabinet Dataset

In [ ]:
!python 04_download_dataset.py

## 4) Augment with Handle Features (05b)

This is required by default in training (`require_augmented: true`).

In [ ]:
!python 05b_augment_handle_data.py

## 5) Quick Sanity Training Run

Use this first to validate your pipeline quickly.

In [ ]:
!python 06_train_policy.py --config configs/diffusion_policy.yaml --epochs 5 --max_episodes 40 --batch_size 32

## 6) Full Training (Optional)

Uncomment and run after sanity training if you want a stronger checkpoint.

In [ ]:
# !python 06_train_policy.py --config configs/diffusion_policy.yaml

## 7) Evaluate Trained Policy

In [ ]:
!python 07_evaluate_policy.py \
  --checkpoint /tmp/cabinet_policy_checkpoints/best_policy.pt \
  --num_rollouts 20 \
  --split pretrain

## 8) Render Rollout Video

In [ ]:
!python 08_visualize_policy_rollout.py \
  --checkpoint /tmp/cabinet_policy_checkpoints/best_policy.pt \
  --offscreen \
  --video_path /tmp/policy_rollout.mp4

from IPython.display import Video, display
display(Video('/tmp/policy_rollout.mp4', embed=True))

## Notes

- Training script enforces augmented data by default.
- If needed, you can override with `--allow_raw` (not recommended).
- Success metric in eval is one-door-open success.